# AWP Tutorial 02: Workflow Loading, Saving & CLI Tools

This notebook teaches how to load, inspect, modify, create, and serialize AWP workflows
programmatically -- plus how to use AWP's built-in CLI tools for packing, validation,
and visualization. **No LLM API key required.**

**Sections:**
1. Loading Workflows
2. Modifying Workflows Programmatically
3. Creating Workflows from Scratch
4. Agent Configuration
5. Safe Expression Evaluator
6. Secrets Loading
7. Template Resolution
8. Pack & Unpack (CLI)
9. Workflow Visualization

In [1]:
import os
from pathlib import Path

# Set working directory to project root
PROJECT_ROOT = Path("/home/shumway/projects/agent-workflow-protocol")
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /home/shumway/projects/agent-workflow-protocol


In [2]:
# Core imports used throughout this notebook
import yaml
import json
import tempfile
import subprocess
from pprint import pprint

from awp.parser import parse_manifest, parse_agent, resolve_templates
from awp.runtime.expressions import safe_eval
from awp.runtime.secrets import load_secrets
from awp.models import (
    AWPManifest, AWPAgent,
    AWPOrchestrationConfig, GraphNode, AWPExecutionConfig,
    StateModel, SharingConfig, PersistenceConfig,
    MemoryConfig, ObservabilityConfig,
    DelegationLoopConfig, DelegationBudget, WorkerPolicy, ValidationConfig,
    SecurityConfig, CircuitBreakerConfig,
)

print("All imports successful.")

All imports successful.


---
## 1. Loading Workflows

AWP workflows are defined in `workflow.awp.yaml` files.  The `parse_manifest()` function
reads them into validated Pydantic models.  Let's load examples 01 through 06 and
explore all 7 semantic layers.

In [3]:
# Define the example workflows we want to load
example_dirs = [
    "examples/01-hello-world",
    "examples/02-research-pipeline",
    "examples/03-chat-team",
    "examples/04-memory-workflow",
    "examples/05-observable-analytics",
    "examples/06-enterprise",
]

manifests = {}
for d in example_dirs:
    wf_path = PROJECT_ROOT / d / "workflow.awp.yaml"
    if wf_path.exists():
        m = parse_manifest(wf_path)
        manifests[d] = m
        print(f"Loaded: {m.workflow.name} (v{m.workflow.version}) -- {m.workflow.description[:60]}...")
    else:
        print(f"  [skip] {wf_path} not found")

print(f"\nTotal workflows loaded: {len(manifests)}")

Loaded: hello-world (v1.0.0) -- Simple greeting workflow demonstrating A0 Prescribed autonom...
Loaded: research-pipeline (v1.0.0) -- Multi-agent research pipeline demonstrating A1 Adaptive auto...
Loaded: chat-team (v1.0.0) -- Agent team with message bus communication demonstrating A1 A...
Loaded: memory-workflow (v1.0.0) -- Memory-enabled research workflow demonstrating A1 Adaptive a...
Loaded: observable-analytics (v1.0.0) -- Analytics pipeline with full observability demonstrating A1 ...
Loaded: enterprise (v1.0.0) -- Full enterprise workflow demonstrating A1 Adaptive autonomy ...

Total workflows loaded: 6


### 1a. Layer 0 -- Manifest (Root Metadata)

Every workflow starts with `awp` (spec version) and `workflow` (name, version, description, tags).

In [4]:
# Pick the hello-world manifest as our primary example
hello = manifests["examples/01-hello-world"]

print("=== Layer 0: Manifest ===")
print(f"  AWP spec version : {hello.awp}")
print(f"  Workflow name    : {hello.workflow.name}")
print(f"  Version          : {hello.workflow.version}")
print(f"  Description      : {hello.workflow.description}")
print(f"  Tags             : {hello.workflow.tags}")
print(f"  Author           : {hello.workflow.author}")

=== Layer 0: Manifest ===
  AWP spec version : 1.0.0
  Workflow name    : hello-world
  Version          : 1.0.0
  Description      : Simple greeting workflow demonstrating A0 Prescribed autonomy level
  Tags             : ['example', 'a0', 'prescribed']
  Author           : AWP Examples


### 1b. Layer 5 -- Orchestration (DAG Graph)

The orchestration layer defines the agent execution graph.

In [5]:
# Show orchestration for the research pipeline (has multiple nodes)
research = manifests["examples/02-research-pipeline"]
orch = research.orchestration

print("=== Layer 5: Orchestration ===")
print(f"  Engine           : {orch.engine}")
print(f"  Execution mode   : {orch.execution.mode}")
print(f"  Timeout (total)  : {orch.execution.timeout.total}s")
print(f"  Max parallel     : {orch.execution.max_parallel_agents}")
print(f"  Error handling   : {orch.execution.error_handling.default}")
print(f"\n  Graph nodes ({len(orch.graph)}):")
for node in orch.graph:
    deps = node.depends_on if node.depends_on else "(none -- root)"
    print(f"    {node.id:20s} agent={node.agent:15s} depends_on={deps}")
    if node.share_output:
        print(f"{'':24s} share_output={node.share_output}")

=== Layer 5: Orchestration ===
  Engine           : dag
  Execution mode   : sequential
  Timeout (total)  : 300s
  Max parallel     : 4
  Error handling   : continue

  Graph nodes (3):
    planner              agent=planner         depends_on=(none -- root)
                         share_output=['research_questions', 'search_strategy']
    researcher           agent=researcher      depends_on=['planner']
                         share_output=['findings', 'sources']
    writer               agent=writer          depends_on=['researcher']
                         share_output=['report']


### 1c. Layer 4 -- State Management

State defines how data flows between agents: sharing strategy, persistence, and required fields.

In [6]:
print("=== Layer 4: State ===")
for name, m in manifests.items():
    short_name = name.split("/")[-1]
    if m.state:
        st = m.state
        print(f"  {short_name:30s} model={st.model:15s} sharing={st.sharing.strategy}")
    else:
        print(f"  {short_name:30s} (no state config)")

=== Layer 4: State ===
  01-hello-world                 model=shared_dict     sharing=full
  02-research-pipeline           model=shared_dict     sharing=selective
  03-chat-team                   model=shared_dict     sharing=selective
  04-memory-workflow             model=shared_dict     sharing=selective
  05-observable-analytics        model=shared_dict     sharing=selective
  06-enterprise                  model=shared_dict     sharing=selective


### 1d. Layer 4 -- Memory

Multi-tier memory: long-term (MEMORY.md), daily logs, episodic, and semantic (vector DB).

In [7]:
print("=== Layer 4: Memory ===")
for name, m in manifests.items():
    short_name = name.split("/")[-1]
    if m.memory:
        mem = m.memory
        # Memory may be a MemoryConfig or a raw dict depending on how the example defines it
        if hasattr(mem, 'enabled'):
            print(f"  {short_name:30s} enabled={mem.enabled}")
        elif isinstance(mem, dict):
            print(f"  {short_name:30s} config keys={list(mem.keys())}")
        else:
            print(f"  {short_name:30s} type={type(mem).__name__}")
    else:
        print(f"  {short_name:30s} (no memory config)")

=== Layer 4: Memory ===
  01-hello-world                 (no memory config)
  02-research-pipeline           (no memory config)
  03-chat-team                   (no memory config)
  04-memory-workflow             enabled=True
  05-observable-analytics        enabled=True
  06-enterprise                  enabled=True


### 1e. Layer 3 -- Communication

Message bus, channels, and communication patterns between agents.

In [8]:
print("=== Layer 3: Communication ===")
for name, m in manifests.items():
    short_name = name.split("/")[-1]
    if m.communication:
        comm = m.communication
        if hasattr(comm, 'bus'):
            num_channels = len(comm.channels) if hasattr(comm, 'channels') else 0
            print(f"  {short_name:30s} bus={comm.bus.type} channels={num_channels}")
            for ch in comm.channels:
                print(f"    {'':28s} channel: {ch.name} ({ch.type})")
        elif isinstance(comm, dict):
            print(f"  {short_name:30s} config keys={list(comm.keys())}")
    else:
        print(f"  {short_name:30s} (no communication config)")

=== Layer 3: Communication ===
  01-hello-world                 (no communication config)
  02-research-pipeline           (no communication config)
  03-chat-team                   bus=internal channels=2
                                 channel: task_assignments (direct)
                                 channel: status_updates (broadcast)
  04-memory-workflow             bus=internal channels=0
  05-observable-analytics        bus=internal channels=0
  06-enterprise                  bus=internal channels=3
                                 channel: alerts (broadcast)
                                 channel: metrics_feed (direct)
                                 channel: analysis_results (direct)


### 1f. Layer 6 -- Observability

Logging, metrics, tracing, audit, and health checks.

In [9]:
print("=== Layer 6: Observability ===")
for name, m in manifests.items():
    short_name = name.split("/")[-1]
    if m.observability:
        obs = m.observability
        if hasattr(obs, 'logging'):
            print(f"  {short_name:30s} logging={obs.logging.level if hasattr(obs.logging, 'level') else '?'}")
            if hasattr(obs, 'metrics') and hasattr(obs.metrics, 'enabled'):
                print(f"    {'':28s} metrics={obs.metrics.enabled} tracing={obs.tracing.enabled}")
        elif isinstance(obs, dict):
            print(f"  {short_name:30s} config keys={list(obs.keys())}")
    else:
        print(f"  {short_name:30s} (no observability config)")

=== Layer 6: Observability ===
  01-hello-world                 (no observability config)
  02-research-pipeline           (no observability config)
  03-chat-team                   (no observability config)
  04-memory-workflow             (no observability config)
  05-observable-analytics        logging=INFO
                                 metrics=True tracing=True
  06-enterprise                  logging=INFO
                                 metrics=True tracing=True


### 1g. Security Layer

Circuit breakers, rate limiting, access control, and secrets management.

In [10]:
print("=== Security ===")
for name, m in manifests.items():
    short_name = name.split("/")[-1]
    if m.security:
        sec = m.security
        if hasattr(sec, 'circuit_breaker'):
            print(f"  {short_name:30s} circuit_breaker={sec.circuit_breaker.enabled}")
            print(f"    {'':28s} rate_limit={sec.rate_limit.enabled}")
            if sec.access_control:
                print(f"    {'':28s} access_control={sec.access_control.enabled}")
        elif isinstance(sec, dict):
            print(f"  {short_name:30s} config keys={list(sec.keys())}")
    else:
        print(f"  {short_name:30s} (no security config)")

=== Security ===
  01-hello-world                 (no security config)
  02-research-pipeline           (no security config)
  03-chat-team                   (no security config)
  04-memory-workflow             (no security config)
  05-observable-analytics        (no security config)
  06-enterprise                  circuit_breaker=True
                                 rate_limit=True
                                 access_control=True


### 1h. Full Manifest Dump

Use Pydantic's `model_dump()` to see the complete manifest as a Python dict.

In [11]:
# Dump the enterprise workflow as a dict to see all layers at once
enterprise = manifests["examples/06-enterprise"]
full_dump = enterprise.model_dump(exclude_none=True)

print("Enterprise workflow -- all top-level keys:")
for key in full_dump:
    val = full_dump[key]
    if isinstance(val, dict):
        print(f"  {key}: dict with {len(val)} keys -> {list(val.keys())[:5]}")
    elif isinstance(val, list):
        print(f"  {key}: list with {len(val)} items")
    else:
        print(f"  {key}: {val}")

Enterprise workflow -- all top-level keys:
  awp: 1.0.0
  workflow: dict with 9 keys -> ['name', 'version', 'description', 'author', 'tags']
  orchestration: dict with 4 keys -> ['engine', 'graph', 'execution', 'subworkflows']
  state: dict with 6 keys -> ['model', 'persistence', 'sharing', 'required_fields', 'auto_inject']
  memory: dict with 9 keys -> ['enabled', 'workspace_dir', 'long_term', 'daily_log', 'episodic']
  communication: dict with 4 keys -> ['bus', 'channels', 'patterns', 'default_channel']
  observability: dict with 5 keys -> ['logging', 'metrics', 'tracing', 'audit', 'health']
  security: dict with 6 keys -> ['circuit_breaker', 'rate_limit', 'access_control', 'secrets_backend', 'audit_security_events']


---
## 2. Modifying Workflows Programmatically

Once loaded, AWP manifests are Pydantic models. You can modify fields, add nodes
to the graph, and serialize back to YAML.

In [12]:
# Load a workflow and modify it
original = parse_manifest(PROJECT_ROOT / "examples/02-research-pipeline/workflow.awp.yaml")

print("BEFORE modification:")
print(f"  Name: {original.workflow.name}")
print(f"  Nodes: {[n.id for n in original.orchestration.graph]}")

# Modify the workflow name
original.workflow.name = "research-pipeline-v2"
original.workflow.description = "Enhanced research pipeline with review step"

# Add a new node to the graph
review_node = GraphNode(
    id="reviewer",
    agent="reviewer",
    depends_on=["writer"],
    share_output=["review_feedback", "approval"],
    description="Reviews the final report for quality",
)
original.orchestration.graph.append(review_node)

print("\nAFTER modification:")
print(f"  Name: {original.workflow.name}")
print(f"  Nodes: {[n.id for n in original.orchestration.graph]}")

BEFORE modification:
  Name: research-pipeline
  Nodes: ['planner', 'researcher', 'writer']

AFTER modification:
  Name: research-pipeline-v2
  Nodes: ['planner', 'researcher', 'writer', 'reviewer']


In [13]:
# Convert the modified manifest back to a dict and then to YAML
modified_dict = original.model_dump(exclude_none=True)

# Write to a temp file
tmp_dir = Path(tempfile.mkdtemp(prefix="awp_tutorial_"))
output_path = tmp_dir / "workflow.awp.yaml"

with open(output_path, "w") as f:
    yaml.dump(modified_dict, f, default_flow_style=False, sort_keys=False)

print(f"Saved modified workflow to: {output_path}")
print(f"\n--- YAML content (first 40 lines) ---")
lines = output_path.read_text().splitlines()
for line in lines[:40]:
    print(line)

Saved modified workflow to: /tmp/awp_tutorial_ij2a1pl6/workflow.awp.yaml

--- YAML content (first 40 lines) ---
awp: 1.0.0
workflow:
  name: research-pipeline-v2
  version: 1.0.0
  description: Enhanced research pipeline with review step
  author: AWP Examples
  tags:
  - example
  - a1
  - adaptive
  - research
  runtime:
    python: '>=3.10'
    required_providers: []
    required_capabilities: []
  dependencies:
    tools: []
    workflows: []
    skills: []
    python: []
  env:
    required: []
    defaults: {}
  settings:
    llm:
      default_provider: openrouter
      models: {}
      temperature: 0.2
    custom: {}
orchestration:
  engine: dag
  graph:
  - id: planner
    agent: planner
    enabled: true
    depends_on: []
    share_input: {}
    share_output:
    - research_questions
    - search_strategy


In [14]:
# Verify by re-parsing the saved file
reloaded = parse_manifest(output_path)
print("Re-parsed successfully!")
print(f"  Name:  {reloaded.workflow.name}")
print(f"  Nodes: {[n.id for n in reloaded.orchestration.graph]}")
print(f"  New node 'reviewer' share_output: {reloaded.orchestration.graph[-1].share_output}")

Re-parsed successfully!
  Name:  research-pipeline-v2
  Nodes: ['planner', 'researcher', 'writer', 'reviewer']
  New node 'reviewer' share_output: ['review_feedback', 'approval']


---
## 3. Creating Workflows from Scratch

Build a complete `workflow.awp.yaml` programmatically using Pydantic models,
then serialize to YAML.

In [15]:
from awp.models.manifest import WorkflowMetadata, WorkflowSettings, LLMSettings
from awp.models.orchestration import TimeoutConfig, ErrorHandling

# Step 1: Define the orchestration graph
graph = [
    GraphNode(
        id="data_fetcher",
        agent="data_fetcher",
        depends_on=[],
        share_output=["raw_data", "metadata"],
        description="Fetches raw data from sources",
    ),
    GraphNode(
        id="transformer",
        agent="transformer",
        depends_on=["data_fetcher"],
        share_output=["cleaned_data", "transform_log"],
        description="Cleans and transforms the data",
    ),
    GraphNode(
        id="analyzer",
        agent="analyzer",
        depends_on=["transformer"],
        share_output=["analysis_results", "confidence"],
        description="Runs analysis on cleaned data",
    ),
    GraphNode(
        id="reporter",
        agent="reporter",
        depends_on=["analyzer"],
        share_output=["final_report"],
        description="Generates the final report",
    ),
]

print(f"Created {len(graph)} graph nodes:")
for node in graph:
    print(f"  {node.id} -> depends_on={node.depends_on}, shares={node.share_output}")

Created 4 graph nodes:
  data_fetcher -> depends_on=[], shares=['raw_data', 'metadata']
  transformer -> depends_on=['data_fetcher'], shares=['cleaned_data', 'transform_log']
  analyzer -> depends_on=['transformer'], shares=['analysis_results', 'confidence']
  reporter -> depends_on=['analyzer'], shares=['final_report']


In [16]:
# Step 2: Build the orchestration config
orchestration = AWPOrchestrationConfig(
    engine="dag",
    graph=graph,
    execution=AWPExecutionConfig(
        mode="sequential",
        timeout=TimeoutConfig(per_agent=60, total=300),
        max_parallel_agents=2,
        error_handling=ErrorHandling(default="continue", max_retries=2),
    ),
)

print(f"Orchestration engine: {orchestration.engine}")
print(f"Execution mode: {orchestration.execution.mode}")
print(f"Timeout: {orchestration.execution.timeout.per_agent}s per agent, {orchestration.execution.timeout.total}s total")

Orchestration engine: dag
Execution mode: sequential
Timeout: 60s per agent, 300s total


In [17]:
# Step 3: Build state and memory configs
state = StateModel(
    model="shared_dict",
    persistence=PersistenceConfig(
        enabled=True,
        path="data/state",
        format="json",
    ),
    sharing=SharingConfig(
        strategy="selective",
    ),
    max_state_size_mb=25,
)

memory = MemoryConfig(
    enabled=True,
    workspace_dir="workspace",
)

print(f"State model: {state.model}, sharing: {state.sharing.strategy}")
print(f"Memory enabled: {memory.enabled}, workspace: {memory.workspace_dir}")

State model: shared_dict, sharing: selective
Memory enabled: True, workspace: workspace


In [18]:
# Step 4: Assemble the full manifest
manifest = AWPManifest(
    awp="1.0.0",
    workflow=WorkflowMetadata(
        name="data-pipeline",
        version="1.0.0",
        description="Automated data pipeline: fetch, transform, analyze, report",
        author="AWP Tutorial",
        tags=["tutorial", "pipeline", "a1"],
    ),
    orchestration=orchestration,
    state=state,
    memory=memory,
)

print(f"Manifest created: {manifest.workflow.name} v{manifest.workflow.version}")
print(f"  Description: {manifest.workflow.description}")
print(f"  Tags: {manifest.workflow.tags}")

Manifest created: data-pipeline v1.0.0
  Description: Automated data pipeline: fetch, transform, analyze, report
  Tags: ['tutorial', 'pipeline', 'a1']


In [19]:
# Step 5: Serialize to YAML and save
scratch_dir = Path(tempfile.mkdtemp(prefix="awp_scratch_"))
scratch_yaml = scratch_dir / "workflow.awp.yaml"

manifest_dict = manifest.model_dump(exclude_none=True)
with open(scratch_yaml, "w") as f:
    yaml.dump(manifest_dict, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {scratch_yaml}")
print(f"\n--- Full YAML ---")
print(scratch_yaml.read_text())

Saved to: /tmp/awp_scratch_vb5vw7d5/workflow.awp.yaml

--- Full YAML ---
awp: 1.0.0
workflow:
  name: data-pipeline
  version: 1.0.0
  description: 'Automated data pipeline: fetch, transform, analyze, report'
  author: AWP Tutorial
  tags:
  - tutorial
  - pipeline
  - a1
  runtime:
    python: '>=3.10'
    required_providers: []
    required_capabilities: []
  dependencies:
    tools: []
    workflows: []
    skills: []
    python: []
  env:
    required: []
    defaults: {}
  settings:
    llm:
      default_provider: openrouter
      models: {}
      temperature: 0.2
    custom: {}
orchestration:
  engine: dag
  graph:
  - id: data_fetcher
    agent: data_fetcher
    enabled: true
    depends_on: []
    share_input: {}
    share_output:
    - raw_data
    - metadata
    description: Fetches raw data from sources
    on_failure: continue
    retry: 0
  - id: transformer
    agent: transformer
    enabled: true
    depends_on:
    - data_fetcher
    share_input: {}
    share_output:
 

In [20]:
# Step 6: Verify by re-parsing
verified = parse_manifest(scratch_yaml)
print("Re-parse verification:")
print(f"  Name:   {verified.workflow.name}")
print(f"  Nodes:  {[n.id for n in verified.orchestration.graph]}")
print(f"  State:  model={verified.state.model}, sharing={verified.state.sharing.strategy}")
print(f"  Memory: enabled={verified.memory.enabled}")
print("  Round-trip OK!")

Re-parse verification:
  Name:   data-pipeline
  Nodes:  ['data_fetcher', 'transformer', 'analyzer', 'reporter']
  State:  model=shared_dict, sharing=selective
  Memory: enabled=True
  Round-trip OK!


---
## 4. Agent Configuration

Agents are defined in `agent.awp.yaml` files. Let's create one programmatically
using the `AWPAgent` Pydantic model.

In [21]:
from awp.models.agent import (
    AgentIdentity, AgentRuntime, ModelConfig, ModelParameters,
    PromptConfig, OutputConfig, OutputField, OutputValidation,
    ReasoningConfig,
)

# Build an agent configuration from scratch
agent = AWPAgent(
    awp_agent="1.0.0",
    identity=AgentIdentity(
        id="data_analyzer",
        role="Data Analysis Specialist",
        description="Analyzes datasets and produces statistical summaries",
        version="1.0.0",
        tags=["analysis", "statistics"],
    ),
    model=ModelConfig(
        provider="openrouter",
        name="",  # Resolved from LLM_MODEL env var at runtime
        parameters=ModelParameters(temperature=0.1, max_tokens=4096),
        reasoning=ReasoningConfig(enabled=True, effort="medium"),
    ),
    prompt=PromptConfig(
        system="prompts/system.md",
        variables={"domain": "financial_analysis"},
    ),
    output=OutputConfig(
        format="json",
        contract={
            "summary": OutputField(type="string", description="Analysis summary", required=True),
            "confidence": OutputField(type="float", description="Confidence score", minimum=0.0, maximum=1.0, required=True),
            "metrics": OutputField(type="object", description="Computed metrics"),
            "recommendations": OutputField(type="array", description="Action items"),
        },
        validation=OutputValidation(mode="strict", on_invalid="retry", max_retries=2),
    ),
)

print(f"Agent: {agent.identity.id}")
print(f"  Role: {agent.identity.role}")
print(f"  Model: provider={agent.model.provider}, name='{agent.model.name}' (runtime-resolved)")
print(f"  Output contract fields: {list(agent.output.contract.keys())}")
print(f"  Reasoning: enabled={agent.model.reasoning.enabled}, effort={agent.model.reasoning.effort}")

Agent: data_analyzer
  Role: Data Analysis Specialist
  Model: provider=openrouter, name='' (runtime-resolved)
  Output contract fields: ['summary', 'confidence', 'metrics', 'recommendations']
  Reasoning: enabled=True, effort=medium


In [22]:
# Save agent.awp.yaml to disk
agent_dir = scratch_dir / "agents" / "data_analyzer"
agent_dir.mkdir(parents=True, exist_ok=True)
agent_yaml_path = agent_dir / "agent.awp.yaml"

agent_dict = agent.model_dump(exclude_none=True)
with open(agent_yaml_path, "w") as f:
    yaml.dump(agent_dict, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {agent_yaml_path}")
print(f"\n--- agent.awp.yaml ---")
print(agent_yaml_path.read_text())

Saved to: /tmp/awp_scratch_vb5vw7d5/agents/data_analyzer/agent.awp.yaml

--- agent.awp.yaml ---
awp_agent: 1.0.0
identity:
  id: data_analyzer
  role: Data Analysis Specialist
  description: Analyzes datasets and produces statistical summaries
  version: 1.0.0
  tags:
  - analysis
  - statistics
runtime:
  class_name: Agent
  strategy_folder: workflow
model:
  provider: openrouter
  name: ''
  parameters:
    temperature: 0.1
    max_tokens: 4096
    top_p: 1.0
  reasoning:
    enabled: true
    effort: medium
    force: true
prompt:
  system: prompts/system.md
  additional: []
  variables:
    domain: financial_analysis
  injection_order:
  - system_prompt
  - skills
  - memory
  - previous_agents
  - user_prompt
  - context
output:
  format: json
  contract:
    summary:
      type: string
      description: Analysis summary
      shareable: true
      sensitive: false
      required: true
      examples: []
    confidence:
      type: float
      description: Confidence score
      

In [23]:
# Load an existing agent from the examples
agents_base = PROJECT_ROOT / "examples/01-hello-world/agents"
agent_dirs = list(agents_base.iterdir()) if agents_base.exists() else []

for ad in agent_dirs:
    agent_file = ad / "agent.awp.yaml"
    if agent_file.exists():
        loaded_agent = parse_agent(agent_file)
        print(f"Loaded agent: {loaded_agent.identity.id}")
        print(f"  Role: {loaded_agent.identity.role}")
        print(f"  Description: {loaded_agent.identity.description}")
        print(f"  Model provider: {loaded_agent.model.provider}")
        print(f"  Prompt system: {loaded_agent.prompt.system}")
        print(f"  Output format: {loaded_agent.output.format}")
    else:
        print(f"  [skip] No agent.awp.yaml in {ad.name}")

Loaded agent: greeter
  Role: greeting_specialist
  Description: Generates personalized greetings based on user input
  Model provider: None
  Prompt system: workflow/instructions/SYSTEM_PROMPT.md
  Output format: json


---
## 5. Safe Expression Evaluator

AWP uses `safe_eval()` for `when` conditions in the orchestration graph.
It uses AST whitelisting -- no `eval()`, no function calls, no imports.

In [24]:
# Basic comparisons
print("=== Comparisons ===")
print(f"  x > 5 and y < 10  (x=7, y=3):  {safe_eval('x > 5 and y < 10', {'x': 7, 'y': 3})}")
print(f"  x > 5 and y < 10  (x=3, y=3):  {safe_eval('x > 5 and y < 10', {'x': 3, 'y': 3})}")
print(f"  score >= 0.8      (score=0.85): {safe_eval('score >= 0.8', {'score': 0.85})}")
print(f"  score >= 0.8      (score=0.5):  {safe_eval('score >= 0.8', {'score': 0.5})}")

=== Comparisons ===
  x > 5 and y < 10  (x=7, y=3):  True
  x > 5 and y < 10  (x=3, y=3):  False
  score >= 0.8      (score=0.85): True
  score >= 0.8      (score=0.5):  False


In [25]:
# Boolean logic
print("=== Boolean Logic ===")
ctx = {"a": True, "b": False, "c": True}
print(f"  a and b:       {safe_eval('a and b', ctx)}")
print(f"  a or b:        {safe_eval('a or b', ctx)}")
print(f"  not b:         {safe_eval('not b', ctx)}")
print(f"  a and (b or c): {safe_eval('a and (b or c)', ctx)}")

=== Boolean Logic ===
  a and b:       False
  a or b:        True
  not b:         True
  a and (b or c): True


In [26]:
# Arithmetic
print("=== Arithmetic ===")
ctx = {"x": 10, "y": 3}
print(f"  x + y:   {safe_eval('x + y', ctx)}")
print(f"  x * y:   {safe_eval('x * y', ctx)}")
print(f"  x // y:  {safe_eval('x // y', ctx)}")
print(f"  x % y:   {safe_eval('x % y', ctx)}")
print(f"  -x:      {safe_eval('-x', ctx)}")

=== Arithmetic ===
  x + y:   13
  x * y:   30
  x // y:  3
  x % y:   1
  -x:      -10


In [27]:
# Attribute and subscript access -- like real workflow state dicts
print("=== Attribute / Subscript Access (Workflow State) ===")

state_ctx = {
    "state": {
        "analyst": {
            "risk_score": 0.72,
            "category": "high",
            "findings": ["anomaly_detected", "threshold_exceeded"],
        },
        "researcher": {
            "confidence": 0.91,
            "sources": 12,
        },
    }
}

# Dotted attribute access on dicts
print(f"  state.analyst.risk_score > 0.3:     {safe_eval('state.analyst.risk_score > 0.3', state_ctx)}")
print(f"  state.analyst.risk_score > 0.9:     {safe_eval('state.analyst.risk_score > 0.9', state_ctx)}")
print(f"  state.researcher.confidence >= 0.9: {safe_eval('state.researcher.confidence >= 0.9', state_ctx)}")

# Subscript access
print(f"  state.researcher.sources > 5:       {safe_eval('state.researcher.sources > 5', state_ctx)}")

# Combined condition -- typical 'when' clause
expr = 'state.analyst.risk_score > 0.5 and state.researcher.confidence > 0.8'
print(f"  {expr}: {safe_eval(expr, state_ctx)}")

=== Attribute / Subscript Access (Workflow State) ===
  state.analyst.risk_score > 0.3:     True
  state.analyst.risk_score > 0.9:     False
  state.researcher.confidence >= 0.9: True
  state.researcher.sources > 5:       True
  state.analyst.risk_score > 0.5 and state.researcher.confidence > 0.8: True


In [28]:
# Show what's BLOCKED -- function calls, imports, etc.
print("=== Blocked Constructs ===")

blocked_exprs = [
    ("print('hello')", "function call"),
    ("__import__('os')", "import"),
    ("lambda x: x", "lambda"),
    ("[x for x in range(5)]", "list comprehension"),
]

for expr, reason in blocked_exprs:
    try:
        safe_eval(expr, {})
        print(f"  {expr:40s} -> ALLOWED (unexpected!)")
    except ValueError as e:
        print(f"  {expr:40s} -> BLOCKED ({reason}): {e}")

=== Blocked Constructs ===
  print('hello')                           -> BLOCKED (function call): Disallowed expression construct: Call
  __import__('os')                         -> BLOCKED (import): Disallowed expression construct: Call
  lambda x: x                              -> BLOCKED (lambda): Disallowed expression construct: Lambda
  [x for x in range(5)]                    -> BLOCKED (list comprehension): Disallowed expression construct: ListComp


---
## 6. Secrets Loading

AWP resolves secrets from multiple sources with priority:
1. `os.environ` (base)
2. `~/.awp/.env` (global fallback)
3. `.env` in workflow dir
4. `secrets.yaml` in workflow dir (highest priority)

In [29]:
# Create a temporary workflow directory with a .env file
secrets_dir = Path(tempfile.mkdtemp(prefix="awp_secrets_"))

# Write a .env file
env_content = """# AWP Tutorial -- example .env file
TUTORIAL_API_KEY=sk-tutorial-12345
DATABASE_URL=postgres://localhost:5432/tutorial_db
DEBUG_MODE=true
"""
(secrets_dir / ".env").write_text(env_content)
print(f"Created .env at: {secrets_dir / '.env'}")
print(f"Contents:\n{env_content}")

Created .env at: /tmp/awp_secrets_y060hhy7/.env
Contents:
# AWP Tutorial -- example .env file
TUTORIAL_API_KEY=sk-tutorial-12345
DATABASE_URL=postgres://localhost:5432/tutorial_db
DEBUG_MODE=true



In [30]:
# Load secrets from that directory
secrets = load_secrets(secrets_dir)

# Show only the keys we set (not all os.environ)
tutorial_keys = ["TUTORIAL_API_KEY", "DATABASE_URL", "DEBUG_MODE"]
print("Loaded secrets (from .env):")
for key in tutorial_keys:
    val = secrets.get(key, "(not found)")
    print(f"  {key} = {val}")

print(f"\nTotal secrets loaded (including os.environ): {len(secrets)}")

Loaded secrets (from .env):
  TUTORIAL_API_KEY = sk-tutorial-12345
  DATABASE_URL = postgres://localhost:5432/tutorial_db
  DEBUG_MODE = true

Total secrets loaded (including os.environ): 128


In [31]:
# Demonstrate secrets.yaml with template references
secrets_yaml_content = """secrets:
  OVERRIDE_KEY: "explicit-value-from-yaml"
  DERIVED_URL: "{{ env.DATABASE_URL }}/derived"
"""

# Set DATABASE_URL in the environment so the template can resolve it
os.environ["DATABASE_URL"] = "postgres://localhost:5432/tutorial_db"
(secrets_dir / "secrets.yaml").write_text(secrets_yaml_content)

# Re-load with secrets.yaml present
secrets2 = load_secrets(secrets_dir)

print("Resolution order demo (secrets.yaml > .env > os.environ):")
print(f"  OVERRIDE_KEY = {secrets2.get('OVERRIDE_KEY', '(not found)')}")
print(f"  DERIVED_URL  = {secrets2.get('DERIVED_URL', '(not found)')}")
print(f"  DATABASE_URL = {secrets2.get('DATABASE_URL', '(not found)')}")

# Clean up env var
del os.environ["DATABASE_URL"]

Resolution order demo (secrets.yaml > .env > os.environ):
  OVERRIDE_KEY = explicit-value-from-yaml
  DERIVED_URL  = postgres://localhost:5432/tutorial_db/derived
  DATABASE_URL = postgres://localhost:5432/tutorial_db


---
## 7. Template Resolution

AWP supports `{{variable}}` template placeholders in YAML values.
The `resolve_templates()` function recursively substitutes them.

In [32]:
# Basic template resolution
template_data = {
    "greeting": "Hello, {{user.name}}!",
    "description": "Workflow for {{workflow.name}} v{{workflow.version}}",
}

context = {
    "user": {"name": "Alice"},
    "workflow": {"name": "data-pipeline", "version": "2.0.0"},
}

resolved = resolve_templates(template_data, context)

print("Template resolution:")
print(f"  Before: {template_data}")
print(f"  After:  {resolved}")

Template resolution:
  Before: {'greeting': 'Hello, {{user.name}}!', 'description': 'Workflow for {{workflow.name}} v{{workflow.version}}'}
  After:  {'greeting': 'Hello, Alice!', 'description': 'Workflow for data-pipeline v2.0.0'}


In [33]:
# Nested template resolution -- works in lists and nested dicts
nested_data = {
    "agents": [
        {"name": "{{workflow.name}}-planner", "timeout": "{{settings.timeout}}"},
        {"name": "{{workflow.name}}-executor", "model": "{{settings.model}}"},
    ],
    "metadata": {
        "author": "{{user.name}}",
        "domain": "{{settings.custom.domain}}",
    },
}

context = {
    "workflow": {"name": "analytics"},
    "settings": {"timeout": "120", "model": "gpt-4", "custom": {"domain": "finance"}},
    "user": {"name": "Bob"},
}

resolved = resolve_templates(nested_data, context)

print("Nested template resolution:")
pprint(resolved)

Nested template resolution:
{'agents': [{'name': 'analytics-planner', 'timeout': '120'},
            {'model': 'gpt-4', 'name': 'analytics-executor'}],
 'metadata': {'author': 'Bob', 'domain': 'finance'}}


In [34]:
# Unresolved templates are preserved as-is (no error)
partial_data = {
    "known": "{{workflow.name}}",
    "unknown": "{{missing.variable}}",
}

context = {"workflow": {"name": "test"}}
resolved = resolve_templates(partial_data, context)

print("Partial resolution (unknown kept as-is):")
print(f"  known:   {resolved['known']}")
print(f"  unknown: {resolved['unknown']}")

Partial resolution (unknown kept as-is):
  known:   test
  unknown: {{missing.variable}}


---
## 8. Pack & Unpack (CLI)

AWP provides `awp pack` and `awp unpack` commands to create and extract
`.awp.zip` archives.  We can call these via subprocess or use the Python
functions directly.

In [35]:
# Use the Python API directly for packing
from awp.packager import pack_workflow, unpack_workflow

hello_dir = PROJECT_ROOT / "examples/01-hello-world"
pack_output_dir = Path(tempfile.mkdtemp(prefix="awp_pack_"))
archive_path = pack_output_dir / "hello-world.awp.zip"

result = pack_workflow(hello_dir, archive_path)
print(f"Packed workflow to: {result}")
print(f"Archive size: {result.stat().st_size} bytes")

Packed workflow to: /tmp/awp_pack_th1xstdg/hello-world.awp.zip
Archive size: 3696 bytes


In [36]:
# Inspect the archive contents
import zipfile

with zipfile.ZipFile(archive_path, "r") as zf:
    print("Archive contents:")
    for info in zf.infolist():
        print(f"  {info.filename:50s} {info.file_size:>8d} bytes")
    
    # Read the package manifest
    manifest_json = json.loads(zf.read("manifest.json"))
    print(f"\nPackage manifest:")
    pprint(manifest_json)

Archive contents:
  agents/greeter/agent.awp.yaml                           880 bytes
  agents/greeter/agent.py                                 805 bytes
  agents/greeter/workflow/instructions/SYSTEM_PROMPT.md      288 bytes
  agents/greeter/workflow/output_schema/output_schema.json      543 bytes
  agents/greeter/workflow/output_schema_desc/output_schema_desc.json      199 bytes
  agents/greeter/workflow/prompt/00_INTRO.md               62 bytes
  workflow.awp.yaml                                       610 bytes
  manifest.json                                           303 bytes
  checksums.sha256                                        754 bytes

Package manifest:
{'awp_package': '1.0.0',
 'checksum': '13b75b578ef072177fb7783b40c16337bc5e263eca6f6a6586b765995621f88a',
 'contents': {'agents': 1,
              'has_memory_snapshot': False,
              'skills': 0,
              'tools': 0},
 'created_at': '2026-03-28T03:15:38.327652+00:00',
 'workflow_name': '01-hello-world'}


In [37]:
# Unpack the archive
unpack_dir = pack_output_dir / "unpacked"
unpacked = unpack_workflow(archive_path, unpack_dir)

print(f"Unpacked to: {unpacked}")
print(f"\nExtracted files:")
for f in sorted(unpacked.rglob("*")):
    if f.is_file():
        rel = f.relative_to(unpacked)
        print(f"  {rel}")

Unpacked to: /tmp/awp_pack_th1xstdg/unpacked

Extracted files:
  agents/greeter/agent.awp.yaml
  agents/greeter/agent.py
  agents/greeter/workflow/instructions/SYSTEM_PROMPT.md
  agents/greeter/workflow/output_schema/output_schema.json
  agents/greeter/workflow/output_schema_desc/output_schema_desc.json
  agents/greeter/workflow/prompt/00_INTRO.md
  workflow.awp.yaml


In [38]:
# Verify the unpacked workflow is valid by re-parsing it
unpacked_manifest = parse_manifest(unpacked / "workflow.awp.yaml")
print(f"Unpacked workflow is valid!")
print(f"  Name: {unpacked_manifest.workflow.name}")
print(f"  Nodes: {[n.id for n in unpacked_manifest.orchestration.graph]}")

Unpacked workflow is valid!
  Name: hello-world
  Nodes: ['greeter']


In [39]:
# Also demonstrate the CLI interface
print("=== CLI: awp pack ===")
cli_archive = pack_output_dir / "hello-cli.awp.zip"
result = subprocess.run(
    ["awp", "pack", str(hello_dir), "-o", str(cli_archive)],
    capture_output=True, text=True
)
print(f"stdout: {result.stdout.strip()}")
if result.returncode != 0:
    print(f"stderr: {result.stderr.strip()}")
print(f"Return code: {result.returncode}")

=== CLI: awp pack ===


stdout: [ok] Packed: /tmp/awp_pack_th1xstdg/hello-cli.awp.zip
Return code: 0


---
## 9. Workflow Visualization

AWP can render orchestration graphs as Mermaid diagrams or ASCII art.

In [40]:
from awp.visualizer import to_mermaid, to_ascii

# Visualize the research pipeline (has interesting dependencies)
research = manifests["examples/02-research-pipeline"]

print("=== Mermaid Diagram: research-pipeline ===")
mermaid_text = to_mermaid(research.orchestration)
print(mermaid_text)
print("\n(Copy the above into https://mermaid.live to render)")

=== Mermaid Diagram: research-pipeline ===
graph TD
    planner[planner]
    researcher[researcher]
    planner --> researcher
    writer[writer]
    researcher --> writer

(Copy the above into https://mermaid.live to render)


In [41]:
# ASCII visualization
print("=== ASCII DAG: research-pipeline ===")
ascii_text = to_ascii(research.orchestration)
print(ascii_text)

=== ASCII DAG: research-pipeline ===
  Level 0: [planner]
  Level 1: [researcher]
  Level 2: [writer]


In [42]:
# Visualize the enterprise workflow (more complex graph with branching)
enterprise = manifests["examples/06-enterprise"]

print("=== Mermaid Diagram: enterprise ===")
print(to_mermaid(enterprise.orchestration))
print()
print("=== ASCII DAG: enterprise ===")
print(to_ascii(enterprise.orchestration))

=== Mermaid Diagram: enterprise ===
graph TD
    data_collector[data_collector]
    code_executor[code_executor]
    data_collector --> code_executor
    analyst[analyst]
    data_collector --> analyst
    communicator[communicator]
    code_executor --> communicator
    analyst --> communicator
    report_writer[report_writer]
    communicator -->|success| report_writer

=== ASCII DAG: enterprise ===
  Level 0: [data_collector]
  Level 1: [analyst, code_executor]  <- parallel
  Level 2: [communicator]
  Level 3: [report_writer]


In [43]:
# Visualize our scratch workflow (created in Section 3)
print("=== Mermaid Diagram: data-pipeline (from scratch) ===")
print(to_mermaid(verified.orchestration))
print()
print("=== ASCII DAG: data-pipeline (from scratch) ===")
print(to_ascii(verified.orchestration))

=== Mermaid Diagram: data-pipeline (from scratch) ===
graph TD
    data_fetcher[data_fetcher]
    transformer[transformer]
    data_fetcher --> transformer
    analyzer[analyzer]
    transformer --> analyzer
    reporter[reporter]
    analyzer --> reporter

=== ASCII DAG: data-pipeline (from scratch) ===
  Level 0: [data_fetcher]
  Level 1: [transformer]
  Level 2: [analyzer]
  Level 3: [reporter]


In [44]:
# CLI visualization
print("=== CLI: awp visualize --format mermaid ===")
result = subprocess.run(
    ["awp", "visualize", str(PROJECT_ROOT / "examples/06-enterprise"), "--format", "mermaid"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"stderr: {result.stderr.strip()}")

=== CLI: awp visualize --format mermaid ===


graph TD
    data_collector[data_collector]
    code_executor[code_executor]
    data_collector --> code_executor
    analyst[analyst]
    data_collector --> analyst
    communicator[communicator]
    code_executor --> communicator
    analyst --> communicator
    report_writer[report_writer]
    communicator -->|success| report_writer



In [45]:
# Bonus: CLI validation
print("=== CLI: awp validate ===")
result = subprocess.run(
    ["awp", "validate", str(PROJECT_ROOT / "examples/02-research-pipeline")],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(f"stderr: {result.stderr.strip()}")
print(f"Return code: {result.returncode}")

=== CLI: awp validate ===


[ok] Manifest parsed: research-pipeline v1.0.0
[ok] Agent parsed: planner
[ok] Agent parsed: researcher
[ok] Agent parsed: writer
[ok] Graph valid
[ok] Contracts valid
[ok] Rules passed

Validation passed for research-pipeline

Return code: 0


In [46]:
# Clean up temporary directories
import shutil

for d in [tmp_dir, scratch_dir, secrets_dir, pack_output_dir]:
    if d.exists():
        shutil.rmtree(d)
        print(f"Cleaned up: {d}")

print("\nAll temporary files removed. Tutorial complete!")

Cleaned up: /tmp/awp_tutorial_ij2a1pl6
Cleaned up: /tmp/awp_scratch_vb5vw7d5
Cleaned up: /tmp/awp_secrets_y060hhy7
Cleaned up: /tmp/awp_pack_th1xstdg

All temporary files removed. Tutorial complete!


---
## Summary

In this tutorial you learned how to:

1. **Load** existing AWP workflows with `parse_manifest()` and inspect all 7 semantic layers
2. **Modify** workflows programmatically -- change fields, add graph nodes, serialize back to YAML
3. **Create** workflows from scratch using Pydantic models (`AWPManifest`, `GraphNode`, etc.)
4. **Configure agents** with `AWPAgent` -- identity, model, prompt, output contract
5. **Evaluate expressions** safely with `safe_eval()` -- the engine behind `when` conditions
6. **Load secrets** from `.env`, `secrets.yaml`, and environment variables
7. **Resolve templates** with `resolve_templates()` for `{{variable}}` substitution
8. **Pack/unpack** workflows into `.awp.zip` archives for distribution
9. **Visualize** workflow DAGs as Mermaid diagrams or ASCII art

None of these operations require an LLM API key -- they work entirely with local parsing,
validation, and serialization.